In [1]:
import torch
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch.nn.functional as F
import torch.nn as nn
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../')
from utilities import load_embedding
from fluProfiler_models import fluProfiler_Config, fluProfiler_HANA
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda:0


In [2]:
dd = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/Sequence/future_H3.csv')
dd_43 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/data_40/Crick_43.csv')

In [11]:
def build_hi_table(data, subtype='H1'):
    """
    构建HI表：serum x virus 矩阵
    返回: HI表DataFrame, serum信息, virus信息
    """
    # 获取唯一的serum和virus
    serum_cols = ['seq_id_a', 'seq_id_b', 'serumPassCat', 'serumName']
    virus_cols = ['seq_id_c', 'seq_id_d', 'virusPassCat', 'virusName']
    
    # 提取serum信息
    serum_cols_available = [col for col in serum_cols if col in data.columns]
    serum_data = data[serum_cols_available].drop_duplicates().reset_index(drop=True)
    serum_data['serum_id'] = range(len(serum_data))
    
    # 提取virus信息
    virus_cols_available = [col for col in virus_cols if col in data.columns]
    virus_data = data[virus_cols_available].drop_duplicates().reset_index(drop=True)
    virus_data['virus_id'] = range(len(virus_data))
    
    # 合并数据以获取serum_id和virus_id
    data_with_ids = data.merge(serum_data, on=serum_cols_available, how='left')
    data_with_ids = data_with_ids.merge(virus_data, on=virus_cols_available, how='left')
    
    # 构建HI表（使用已有的label值）
    hi_table = data_with_ids.pivot_table(
        values='label', 
        index='serum_id', 
        columns='virus_id', 
        aggfunc='mean'
    )
    
    print(f"{subtype} HI Table shape: {hi_table.shape}")
    print(f"Missing values: {hi_table.isna().sum().sum()} ({hi_table.isna().sum().sum() / (hi_table.shape[0] * hi_table.shape[1]) * 100:.2f}%)")
    
    return hi_table, serum_data, virus_data, data_with_ids

# 构建H1的HI表
h1_hi_table, h1_serum_data, h1_virus_data, h1_data_with_ids = build_hi_table(h1_data, 'H1')

# 构建H3的HI表
h3_hi_table, h3_serum_data, h3_virus_data, h3_data_with_ids = build_hi_table(h3_data, 'H3')


H1 HI Table shape: (67, 5904)
Missing values: 341681 (86.38%)
H3 HI Table shape: (151, 5259)
Missing values: 747312 (94.11%)


In [ ]:
def complete_hi_table(hi_table, serum_data, virus_data, data_with_ids, model, emb_dict, device, subtype='H1'):
    """
    使用fluProfiler模型补全HI表
    """
    # 找到缺失值的位置
    missing_mask = hi_table.isna()
    missing_pairs = []
    
    for serum_idx in range(len(hi_table.index)):
        for virus_idx in range(len(hi_table.columns)):
            if missing_mask.iloc[serum_idx, virus_idx]:
                serum_id = hi_table.index[serum_idx]
                virus_id = hi_table.columns[virus_idx]
                
                # 获取对应的serum和virus信息
                serum_info = serum_data[serum_data['serum_id'] == serum_id].iloc[0]
                virus_info = virus_data[virus_data['virus_id'] == virus_id].iloc[0]
                
                missing_pairs.append({
                    'serum_id': serum_id,
                    'virus_id': virus_id,
                    'serum_info': serum_info,
                    'virus_info': virus_info
                })
    
    if len(missing_pairs) == 0:
        print(f"{subtype}: No missing values to complete!")
        return hi_table
    
    print(f"{subtype}: Found {len(missing_pairs)} missing pairs to predict")
    
    # 构建预测数据
    prediction_data = []
    pair_indices = []  # 保存对应的serum_id和virus_id
    for pair in missing_pairs:
        serum_info = pair['serum_info']
        virus_info = pair['virus_info']
        
        # 构建数据行（需要与训练数据格式一致）
        row = {
            'seq_id_a': serum_info.get('seq_id_a', ''),
            'seq_id_b': serum_info.get('seq_id_b', ''),
            'seq_id_c': virus_info.get('seq_id_c', ''),
            'seq_id_d': virus_info.get('seq_id_d', ''),
            'serumPassCat': serum_info.get('serumPassCat', 'CELL'),
            'virusPassCat': virus_info.get('virusPassCat', 'CELL'),
            'label': 0.0  # 占位符
        }
        prediction_data.append(row)
        pair_indices.append((pair['serum_id'], pair['virus_id']))
    
    pred_df = pd.DataFrame(prediction_data)
    
    # 检查所需的embedding是否存在
    required_embeddings = set()
    for _, row in pred_df.iterrows():
        for col in ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d']:
            if col in row and pd.notna(row[col]):
                required_embeddings.add(f"matrix_{row[col]}")
    
    missing_emb = required_embeddings - set(emb_dict.keys())
    if missing_emb:
        print(f"Warning: {len(missing_emb)} embeddings missing, filtering data...")
        # 创建过滤掩码
        filter_mask = pred_df.apply(lambda row: all(
            f"matrix_{row[col]}" in emb_dict 
            for col in ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d'] 
            if col in row and pd.notna(row[col])
        ), axis=1)
        # 同时过滤pred_df和pair_indices
        pred_df = pred_df[filter_mask].reset_index(drop=True)
        pair_indices = [pair_indices[i] for i in range(len(pair_indices)) if filter_mask.iloc[i]]
        print(f"After filtering: {len(pred_df)} pairs to predict")
    
    if len(pred_df) == 0:
        print(f"{subtype}: No valid pairs after filtering!")
        return hi_table
    
    # 创建DataLoader并预测
    pred_dataset = fluProfiler_Dataset(pred_df)
    pred_dataloader = DataLoader(pred_dataset, batch_size=32, shuffle=False)
    
    predictions = predict_titers(model, pred_dataloader, emb_dict, device)
    
    # 更新HI表
    completed_table = hi_table.copy()
    valid_pairs = []
    
    # 使用pair_indices更新HI表（pair_indices已经与pred_df对齐）
    for idx, (s_id, v_id) in enumerate(pair_indices):
        if idx < len(predictions):
            if s_id in completed_table.index and v_id in completed_table.columns:
                completed_table.loc[s_id, v_id] = predictions[idx]
                valid_pairs.append((s_id, v_id))
    
    print(f"{subtype}: Completed {len(valid_pairs)} predictions")
    return completed_table

# 加载embedding
# 需要根据实际路径调整
embedding_paths = [
    '/mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/data_40/embedding_Crick',
    '/mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/data_40/embedding_41',
    '/mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/data_40/embedding_42',
    '/mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/data_40/embedding_43'
]

emb_dict = {}
for path in embedding_paths:
    if os.path.exists(path):
        print(f"Loading embeddings from: {path}")
        temp_emb = load_embedding(path)
        emb_dict.update(temp_emb)
        print(f"Loaded {len(temp_emb)} embeddings, total: {len(emb_dict)}")

print(f"\nTotal embeddings loaded: {len(emb_dict)}")


Loading embeddings from: /mnt/zzbnew/peixunban/chenyihao/fluProfiler/data/data_40/embedding_Crick


Loading tensor:   2%|▏         | 183/9711 [00:18<16:23,  9.69file/s]


KeyboardInterrupt: 

: 

In [ ]:
# 补全H1的HI表
if h1_data is not None and 'h1_hi_table' in locals():
    print("\nCompleting H1 HI Table...")
    h1_hi_table_completed = complete_hi_table(
        h1_hi_table, h1_serum_data, h1_virus_data, h1_data_with_ids, 
        model, emb_dict, device, 'H1'
    )
    # 填充剩余缺失值（使用均值或中位数）
    h1_hi_table_completed = h1_hi_table_completed.fillna(h1_hi_table_completed.mean().mean())
    print(f"H1 HI Table completed! Shape: {h1_hi_table_completed.shape}")

# 补全H3的HI表
if h3_data is not None and 'h3_hi_table' in locals():
    print("\nCompleting H3 HI Table...")
    h3_hi_table_completed = complete_hi_table(
        h3_hi_table, h3_serum_data, h3_virus_data, h3_data_with_ids, 
        model, emb_dict, device, 'H3'
    )
    # 填充剩余缺失值（使用均值或中位数）
    h3_hi_table_completed = h3_hi_table_completed.fillna(h3_hi_table_completed.mean().mean())
    print(f"H3 HI Table completed! Shape: {h3_hi_table_completed.shape}")


In [ ]:
def create_antigenic_map_3d(hi_table, virus_data, subtype='H1', n_components=3):
    """
    使用MDS对HI表进行降维，创建3D抗原图
    HI表是serum x virus矩阵，我们基于virus之间的抗原距离进行MDS
    """
    # 转置HI表，使virus作为行（样本）
    hi_table_t = hi_table.T
    
    # 计算virus之间的距离矩阵
    # 使用欧氏距离（也可以使用其他距离度量，如曼哈顿距离）
    # 对于抗原距离，通常使用HI值的差异
    distance_matrix = pdist(hi_table_t.values, metric='euclidean')
    distance_matrix_square = squareform(distance_matrix)
    
    # 应用MDS
    mds = MDS(n_components=n_components, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
    mds_coords = mds.fit_transform(distance_matrix_square)
    
    print(f"{subtype} MDS stress: {mds.stress_:.4f}")
    
    return mds_coords, mds

# 为H1创建3D抗原图
if 'h1_hi_table_completed' in locals():
    print("\nCreating H1 3D Antigenic Map...")
    h1_coords, h1_mds = create_antigenic_map_3d(h1_hi_table_completed, h1_virus_data, 'H1')
    print(f"H1 coordinates shape: {h1_coords.shape}")

# 为H3创建3D抗原图
if 'h3_hi_table_completed' in locals():
    print("\nCreating H3 3D Antigenic Map...")
    h3_coords, h3_mds = create_antigenic_map_3d(h3_hi_table_completed, h3_virus_data, 'H3')
    print(f"H3 coordinates shape: {h3_coords.shape}")


In [ ]:
def plot_3d_antigenic_map(coords, virus_data, subtype='H1', title_suffix=''):
    """
    绘制3D抗原图
    """
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # 绘制散点
    scatter = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], 
                        c=range(len(coords)), cmap='viridis', 
                        s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    # 设置标签
    ax.set_xlabel('MDS Dimension 1', fontsize=12)
    ax.set_ylabel('MDS Dimension 2', fontsize=12)
    ax.set_zlabel('MDS Dimension 3', fontsize=12)
    ax.set_title(f'{subtype} Antigenic Evolution Map (3D){title_suffix}', fontsize=14, fontweight='bold')
    
    # 添加颜色条
    cbar = plt.colorbar(scatter, ax=ax, shrink=0.8, pad=0.1)
    cbar.set_label('Virus Index', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# 绘制H1的3D抗原图
if 'h1_coords' in locals():
    print("\nPlotting H1 3D Antigenic Map...")
    h1_fig = plot_3d_antigenic_map(h1_coords, h1_virus_data, 'H1', ' - Completed with fluProfiler')
    
# 绘制H3的3D抗原图
if 'h3_coords' in locals():
    print("\nPlotting H3 3D Antigenic Map...")
    h3_fig = plot_3d_antigenic_map(h3_coords, h3_virus_data, 'H3', ' - Completed with fluProfiler')


In [ ]:
# 可选：保存补全的HI表和坐标
save_dir = '/mnt/zzbnew/peixunban/chenyihao/fluProfiler/src/Section3_ActiveLearning/'

if 'h1_hi_table_completed' in locals():
    h1_hi_table_completed.to_csv(os.path.join(save_dir, 'H1_HI_table_completed.csv'))
    h1_coords_df = pd.DataFrame(h1_coords, columns=['MDS1', 'MDS2', 'MDS3'])
    h1_coords_df.to_csv(os.path.join(save_dir, 'H1_MDS_coordinates.csv'), index=False)
    print("H1 results saved!")

if 'h3_hi_table_completed' in locals():
    h3_hi_table_completed.to_csv(os.path.join(save_dir, 'H3_HI_table_completed.csv'))
    h3_coords_df = pd.DataFrame(h3_coords, columns=['MDS1', 'MDS2', 'MDS3'])
    h3_coords_df.to_csv(os.path.join(save_dir, 'H3_MDS_coordinates.csv'), index=False)
    print("H3 results saved!")
